In [ ]:
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
DB_USER = USERVAR
DB_PASSWORD = PSWDVAR
DB_HOST = HOSTVAR   # MariaDB VM IP
DB_PORT = "3306"
DB_NAME = "budget"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def post_to_sql(df, table):
    df.to_sql(
    table,
    engine,
    if_exists="replace",
    index=False
)

In [ ]:
main = pd.read_csv('main.csv')
main['Account'] = "Main"

In [ ]:
bills = pd.read_csv('bills.csv')
bills['Account'] = "Bills"

In [ ]:
savings = pd.read_csv('savings.csv')
savings['Account'] = "Main Savings"

In [ ]:
full = pd.concat([bills, main, savings])

In [ ]:
full['Date'] = pd.to_datetime(full['Date'])

In [ ]:
df_clean = full[full["Category"] != "Transfer"].copy()

In [ ]:
df_clean["month"] = df_clean["Date"].dt.to_period("M")

In [ ]:
monthly = (
    df_clean
    .groupby("month")
    .agg(
        income=("Amount", lambda x: x[x > 0].sum()),
        expenses=("Amount", lambda x: abs(x[x < 0].sum()))
    )
    .reset_index()
)

monthly["month"] = monthly["month"].astype(str)

In [ ]:
monthly['overunder'] = monthly['income'] - monthly['expenses']

In [ ]:
monthly

In [ ]:
patterns = ["Therapy Trails", "Childcare"]

regex = "|".join(patterns)

filtered_df = df_clean[~df_clean["Description"].str.contains(regex, regex=True, na=False)]
filtered_df

In [ ]:
filtered_monthly = (
    filtered_df
    .groupby("month")
    .agg(
        income=("Amount", lambda x: x[x > 0].sum()),
        expenses=("Amount", lambda x: abs(x[x < 0].sum()))
    )
    .reset_index()
)

filtered_monthly["month"] = filtered_monthly["month"].astype(str)

In [ ]:
filtered_monthly

In [ ]:
filtered_monthly['overunder'] = filtered_monthly['income'] - filtered_monthly['expenses']

In [ ]:
total = monthly['income'].sum() - monthly['expenses'].sum()

In [ ]:
filtered_total = filtered_monthly['income'].sum() - filtered_monthly['expenses'].sum()

In [ ]:
post_to_sql(df_clean, "Clean")

In [ ]:
post_to_sql(full, "Full")

In [ ]:
post_to_sql(filtered_df, "Filtered")

In [ ]:
post_to_sql(filtered_monthly, "Filtered Monthly")

In [ ]:
post_to_sql(monthly, "Monthly")